# Dateien lesen und schreiben in Python

Bisher haben wir Werte immer nur in Variablen gespeichert. Das Problem dabei: Sobald ein Python-Programm beendet wird, sind alle Variablen und ihre Werte unwiderruflich weg, denn der Arbeitsspeicher wird geleert.

Stell dir eine Variable wie einen Notizzettel auf deinem Schreibtisch vor: Solange du im Büro bist, kannst du jederzeit draufschauen und ihn verändern. Räumst du den Schreibtisch am Abend aber komplett auf, ist der Zettel weg. Eine **Datei** ist dagegen wie ein Eintrag in einem Aktenordner: Er bleibt auch dann erhalten, wenn du abends das Licht ausmachst und am nächsten Tag – oder in einem ganz anderen Programm – wieder darauf zugreifen willst.

In dieser Session geht es darum, in Python Textdateien zu lesen und zu schreiben – eine Grundfertigkeit für praktisch jedes Python-Programm, das mit echten Daten arbeitet.

## Lernziele und Inhalt

**Lernziele**

1. Dateien mit `open()` und dem `with`-Statement öffnen und sicher wieder schließen.
2. Dateiinhalte mit `.read()`, `.readline()` und `.readlines()` oder zeilenweise mit `for` einlesen.
3. Mit den Modi `'w'` und `'a'` Dateien neu schreiben bzw. ergänzen.
4. Fehlende Dateien mit `try`/`except FileNotFoundError` abfangen.
5. (Optional) Strukturierte Daten als CSV und JSON speichern und laden sowie JSON-Daten von einer API abrufen.

**Inhalt**

- [1. Dateien lesen](#lesen)
  - [1.1 `open()` und das `with`-Statement](#open-with)
  - [1.2 Lesemethoden: `read()`, `readline()`, `readlines()`](#lesemethoden)
  - [1.3 Zeilenweise iterieren](#iterieren)
- [2. Dateien schreiben](#schreiben)
  - [2.1 Schreiben mit dem Modus `'w'`](#modus-w)
  - [2.2 Anhängen mit dem Modus `'a'`](#modus-a)
- [3. Fehlerbehandlung beim Dateizugriff](#fehlerbehandlung)
- [4. Zusammenfassung](#zusammenfassung)
- [5. Vertiefung: Strukturierte Daten mit CSV und JSON (Optional)](#vertiefung)
  - [5.1 CSV](#csv)
  - [5.2 JSON](#json)
  - [5.3 Datenströme über das Internet](#api)
- [Übungsaufgaben](#uebungsaufgaben)

<a id="lesen"></a>

## 1. Dateien lesen

In diesem Ordner liegt bereits eine Datei `einkaufsliste.txt` (so, als hätte dir ein:e Kolleg:in sie gegeben). Diese Datei lesen wir zunächst ein.

<a id="open-with"></a>

### 1.1 `open()` und das `with`-Statement

Mit der eingebauten Funktion `open()` kannst du eine Datei öffnen. Sie erwartet mindestens den **Dateipfad** (hier reicht der reine Dateiname, da die Datei im selben Verzeichnis wie dieses Notebook liegt) und liefert ein **Datei-Objekt** zurück, über das du auf den Inhalt zugreifen kannst.

⚠️ Eine geöffnete Datei sollte immer auch wieder **geschlossen** werden, mit `.close()`. Andernfalls bleibt sie ggf. für andere Programme gesperrt, oder nicht gespeicherte Änderungen gehen verloren.

In [ ]:
shopping_list_file = open("einkaufsliste.txt", "r", encoding="utf-8")
content = shopping_list_file.read()
shopping_list_file.close()

print(content)

Das Problem an diesem Vorgehen: Tritt zwischen `open()` und `close()` ein Fehler auf, wird `.close()` nie ausgeführt und die Datei bleibt offen. Der sicherere und in Python übliche Weg ist deshalb das `with`-Statement:

In [ ]:
with open("einkaufsliste.txt", "r", encoding="utf-8") as shopping_list_file:
    content = shopping_list_file.read()

print(content)

💡 `with` öffnet die Datei, führt den eingerückten Code-Block aus und schließt die Datei danach **automatisch** — auch dann, wenn im Block ein Fehler auftritt. Wie das genau funktioniert (Stichwort **Context Manager**), ist Thema der Live Session "Erweiterte Python Syntax". Ab hier verwenden wir für Dateizugriffe deshalb ausschließlich `with`.

⚠️ Ohne explizite Angabe von `encoding="utf-8"` verwendet Python je nach Betriebssystem unterschiedliche Standard-Kodierungen. Das kann bei Sonderzeichen wie Umlauten zu Fehlern oder falsch dargestellten Zeichen führen, daher geben wir `encoding` hier immer bewusst mit an.

<a id="lesemethoden"></a>

### 1.2 Lesemethoden: `read()`, `readline()`, `readlines()`

Neben `.read()`, das den kompletten Dateiinhalt als einen einzigen String zurückgibt, gibt es zwei weitere Möglichkeiten, Zeilen aus einer Datei einzulesen:

- **`.readline()`**: Liest genau **eine** Zeile und gibt sie als String zurück. Beim nächsten Aufruf wird jeweils die nächste Zeile gelesen.
- **`.readlines()`**: Liest **alle** Zeilen auf einmal und gibt sie als Liste von Strings zurück.

In [ ]:
with open("einkaufsliste.txt", "r", encoding="utf-8") as shopping_list_file:
    first_line = shopping_list_file.readline()
    second_line = shopping_list_file.readline()

print(repr(first_line))     # repr() macht unsichtbare Zeichen wie \n sichtbar
print(repr(second_line))

⚠️ Beachte, dass jede Zeile das Zeilenumbruchzeichen `\n` am Ende enthält (außer möglicherweise die letzte Zeile der Datei).

In [ ]:
with open("einkaufsliste.txt", "r", encoding="utf-8") as shopping_list_file:
    lines = shopping_list_file.readlines()

print(lines)
print(f"Anzahl der Zeilen: {len(lines)}")

<a id="iterieren"></a>

### 1.3 Zeilenweise iterieren

Am häufigsten willst du eine Datei Zeile für Zeile verarbeiten. Ein Datei-Objekt ist selbst iterierbar — du kannst also direkt mit einer `for`-Schleife darüber iterieren, ohne vorher `.readlines()` aufzurufen:

In [ ]:
with open("einkaufsliste.txt", "r", encoding="utf-8") as shopping_list_file:
    for line in shopping_list_file:
        print(line.strip())  # .strip() entfernt das Zeilenumbruchzeichen am Ende

💡 Diese Variante ist auch bei sehr großen Dateien speicherschonend, da nie die komplette Datei auf einmal im Arbeitsspeicher gehalten werden muss – anders als bei `.read()` oder `.readlines()`.

<a id="schreiben"></a>

## 2. Dateien schreiben

Zum Schreiben in eine Datei übergeben wir `open()` einen zweiten Parameter, den **Modus** (engl. *mode*). Die wichtigsten Modi sind:

- **`'r'`**: Lesen (*read*, Standard-Modus) — die Datei muss bereits existieren.
- **`'w'`**: Schreiben (*write*) — erstellt die Datei neu bzw. **überschreibt** den kompletten bisherigen Inhalt.
- **`'a'`**: Anhängen (*append*) — ergänzt neuen Inhalt am Ende einer bestehenden Datei, ohne den bisherigen Inhalt zu löschen.

<a id="modus-w"></a>

### 2.1 Schreiben mit dem Modus `'w'`

Ein Beispiel, das Lesen und Schreiben kombiniert: Wir lesen die Einkaufsliste ein, wandeln jeden Eintrag in Großbuchstaben um und schreiben das Ergebnis in eine neue Datei.

In [ ]:
with open("einkaufsliste.txt", "r", encoding="utf-8") as input_file:
    items = input_file.readlines()
print(items)

with open("einkaufsliste_neu.txt", "w", encoding="utf-8") as output_file:
    for item in items:
        output_file.write(item.strip().upper() + "\n")

Mit `.write()` schreibst du einen einzelnen String in die Datei. Im Gegensatz zu `print()` fügt `.write()` dabei **keinen** automatischen Zeilenumbruch hinzu — den musst du bei Bedarf selbst ergänzen (`"\n"`).

Zur Kontrolle lesen wir die neue Datei direkt wieder ein:

In [ ]:
with open("einkaufsliste_neu.txt", "r", encoding="utf-8") as new_file:
    print(new_file.read())

Führst du die Schreib-Zelle mehrmals aus, bleibt das Ergebnis trotzdem gleich, da der Modus `'w'` die Datei bei jeder Ausführung komplett neu erstellt (und den alten Inhalt verwirft).

<a id="modus-a"></a>

### 2.2 Anhängen mit dem Modus `'a'`

Willst du dagegen bestehenden Inhalt einer Datei erhalten und nur etwas ergänzen, nutzt du den Modus `'a'`:

In [ ]:
with open("einkaufsliste_neu.txt", "a", encoding="utf-8") as new_file:
    new_file.write("SCHOKOLADE\n")

with open("einkaufsliste_neu.txt", "r", encoding="utf-8") as new_file:
    print(new_file.read())

⚠️ Führst du diese Zelle mehrmals aus, wird `"SCHOKOLADE"` jedes Mal erneut angehängt — anders als beim Modus `'w'` verändert sich das Ergebnis hier also bei jeder erneuten Ausführung.

<a id="fehlerbehandlung"></a>

## 3. Fehlerbehandlung beim Dateizugriff

Du kennst `try`/`except` bereits aus der Session "Fehlerbehandlung". Beim Umgang mit Dateien ist Fehlerbehandlung besonders wichtig, da du selten sicher wissen kannst, ob eine Datei tatsächlich existiert oder les- bzw. schreibbar ist — zum Beispiel, weil sie von einem anderen Programm gelöscht, umbenannt oder noch gar nicht angelegt wurde.

Der häufigste Fehler beim Lesen ist `FileNotFoundError`:

In [ ]:
try:
    with open("datei_die_nicht_existiert.txt", "r", encoding="utf-8") as missing_file:
        missing_file.read()
except FileNotFoundError:
    print("Die Datei wurde nicht gefunden.")

Ohne das `try`/`except` würde dieser Fehler das komplette Programm abbrechen. Praktisch nutzt du den `except`-Block oft, um kontrolliert zu reagieren — zum Beispiel, indem du bei einer fehlenden Datei einen Standardwert verwendest und die Datei direkt neu anlegst:

In [ ]:
try:
    with open("konfiguration.txt", "r", encoding="utf-8") as config_file:
        config_content = config_file.read()
except FileNotFoundError:
    print("Keine Konfigurationsdatei gefunden, lege eine neue mit Standardwerten an.")
    config_content = "modus=standard\n"
    with open("konfiguration.txt", "w", encoding="utf-8") as config_file:
        config_file.write(config_content)

print(config_content)

Beim ersten Ausführen existiert `konfiguration.txt` noch nicht, daher läuft der `except`-Block. Ab dem zweiten Ausführen ist die Datei vorhanden und wird ohne Meldung eingelesen.

<a id="zusammenfassung"></a>

## 4. Zusammenfassung

Dateien speichern Daten über das Programmende hinaus. In Python öffnet `open()` eine Datei, das `with`-Statement sorgt dafür, dass sie anschließend wieder geschlossen wird. Der Modus legt fest, ob gelesen, neu geschrieben oder angehängt wird.

### Behandelte Konzepte

1. Dateien mit `open(<pfad>, <modus>, encoding="utf-8")` und `with ... as` öffnen und automatisch schließen lassen.
2. Den Inhalt mit `.read()` als String, mit `.readline()` zeilenweise oder mit `.readlines()` als Liste einlesen.
3. Mit `for line in <datei>:` speicherschonend über alle Zeilen iterieren und `\n` mit `.strip()` entfernen.
4. Mit `'w'` eine Datei neu schreiben und mit `'a'` Inhalt am Ende ergänzen.
5. Mit `.write(text)` Strings schreiben und Zeilenumbrüche selbst ergänzen.
6. Fehlende Dateien mit `try`/`except FileNotFoundError` abfangen, statt das Programm abbrechen zu lassen.

### Kurzreferenz

| Schreibweise | Bedeutung |
|---|---|
| `with open("datei.txt", "r", encoding="utf-8") as file:` | Datei lesen, danach automatisch schließen |
| `file.read()` | kompletter Inhalt als String |
| `file.readline()` | nächste Zeile als String |
| `file.readlines()` | alle Zeilen als Liste von Strings |
| `for line in file:` | Zeile für Zeile verarbeiten |
| `open(..., "w", ...)` | neu schreiben, bisheriger Inhalt wird überschrieben |
| `open(..., "a", ...)` | am Ende anhängen |
| `file.write("text\n")` | String schreiben, ohne automatischen Zeilenumbruch |
| `except FileNotFoundError:` | fehlende Datei abfangen |

Diese Grundlagen sind die Basis für fast jede Art von Datenaustausch zwischen Programmen — von einfachen Textdateien bis zu strukturierten Formaten wie CSV oder JSON.

<a id="vertiefung"></a>

## 5. Vertiefung: Strukturierte Daten mit CSV und JSON (Optional)

Reine Textdateien sind für einzelne Zeilen praktisch, aber sobald deine Daten aus mehreren **Spalten** oder **verschachtelten Strukturen** bestehen (z.B. Name und E-Mail-Adresse pro Person), stößt du schnell an Grenzen. Für solche Fälle haben sich zwei Formate etabliert, die Python jeweils mit einem eigenen Modul direkt unterstützt: **CSV** und **JSON**.

Ein **Modul** ist eine Sammlung von zusätzlichem Python-Code, den wir per `import` in unser Programm einbinden können (mehr dazu in einer späteren Session).

<a id="csv"></a>

### 5.1 CSV

Das **CSV-Format** (*Comma-Separated Values*) ist im Grunde eine Textdatei, in der jede Zeile eine Reihe und ein Trennzeichen (meist `,`) die Spalten markiert. Es eignet sich für **tabellarische** Daten, also gleichförmige Zeilen mit denselben Spalten — z.B. eine Kontaktliste.

In [ ]:
import csv

contacts = [
    {"name": "Anna", "email": "anna@example.com"},
    {"name": "Bernd", "email": "bernd@example.com"},
    {"name": "Claudia", "email": "claudia@example.com"},
]

with open("kontakte.csv", "w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["name", "email"])
    writer.writeheader()
    writer.writerows(contacts)

💡 Der Parameter `newline=""` verhindert, dass beim Schreiben unter Windows zusätzliche Leerzeilen entstehen — bei CSV-Dateien sollte er deshalb immer gesetzt werden.

Zum Einlesen bietet das Modul passend den `csv.DictReader`, der jede Zeile direkt als Dictionary zurückgibt:

In [ ]:
with open("kontakte.csv", "r", encoding="utf-8", newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    for row in reader:
        print(row)

<a id="json"></a>

### 5.2 JSON

**JSON** (*JavaScript Object Notation*) ist ein weiteres verbreitetes Format für strukturierte Daten, das sich sehr nah an Pythons eigenen Datenstrukturen orientiert: Objekte entsprechen Dictionaries, Listen entsprechen Listen. Anders als CSV eignet sich JSON deshalb auch für **verschachtelte** Daten und wird häufig zum Datenaustausch zwischen Webanwendungen bzw. APIs verwendet.

Python bringt dafür das eingebaute `json`-Modul mit. `json.dump()` schreibt eine Datenstruktur direkt in eine geöffnete Datei:

In [ ]:
import json

with open("kontakte.json", "w", encoding="utf-8") as json_file:
    json.dump(contacts, json_file, indent=2, ensure_ascii=False)

Beim Einlesen liefert `json.load()` die ursprüngliche Datenstruktur zurück — hier also wieder eine Liste von Dictionaries, genau wie unsere `contacts`-Variable:

In [ ]:
with open("kontakte.json", "r", encoding="utf-8") as json_file:
    loaded_contacts = json.load(json_file)

print(loaded_contacts)
print(type(loaded_contacts))

`json.dump()`/`json.load()` arbeiten direkt mit Datei-Objekten. Die Variante `json.dumps()`/`json.loads()` (mit „s" für *string*) wandelt eine Datenstruktur stattdessen in einen String um bzw. aus einem String zurück — praktisch, wenn du JSON-Daten z.B. über eine Netzwerkverbindung sendest statt in eine Datei zu schreiben.

<a id="api"></a>

### 5.3 Datenströme über das Internet

Nicht alle Daten liegen als lokale Datei vor, viele Programme tauschen Daten direkt über das Internet aus, z.B. über offene **APIs** (*Application Programming Interfaces*, Programmierschnittstellen). Eine API stellt Daten über eine URL bereit, die du ähnlich wie eine Webseite abrufen kannst — nur dass die Antwort statt HTML meist **JSON** ist. Das dafür gängige (wenn auch nicht in Python eingebaute) Modul heißt `requests`.

Als Beispiel nutzen wir die offene Wetter-API von Open-Meteo, die ganz ohne Anmeldung oder API-Key funktioniert:

In [ ]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": 51.34, "longitude": 12.37, "current": "temperature_2m"}

response = requests.get(url, params=params)
data = response.json()

print(data)
print(f"Aktuelle Temperatur in Leipzig: {data['current']['temperature_2m']}°C")

`response.json()` macht im Hintergrund nichts anderes als `json.loads(response.text)`. Du bekommst die Antwort also direkt als Python-Datenstruktur zurück, genau wie beim Einlesen einer JSON-Datei mit `json.load()`.

💡 Da Netzwerkzugriffe von vielen externen Faktoren abhängen (Internetverbindung, Server erreichbar, ungültige Anfrage, ...), lohnt sich hier `try`/`except` besonders, z.B. mit `response.raise_for_status()`, das bei einem Fehlerstatus (4xx/5xx) eine Exception auslöst:

In [ ]:
try:
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
except requests.exceptions.RequestException as error:
    print(f"Anfrage fehlgeschlagen: {error}")
else:
    data = response.json()
    print(f"Aktuelle Temperatur in Leipzig: {data['current']['temperature_2m']}°C")

Viele offene APIs liefern Daten im JSON-Format. Das Grundprinzip bleibt dabei immer gleich: URL (ggf. mit Parametern) abrufen, Antwort mit `.json()` auswerten.

**Praxisbeispiel:** Die Stadt Dortmund stellt ihre Geodaten (u.a. Flurstücke und Gebäude aus dem Liegenschaftskataster ALKIS) über eine offene WFS-Schnittstelle als GeoJSON bereit. Genau hier zeigt sich, warum Fehlerbehandlung bei Netzwerkzugriffen so wichtig ist, z.B. wenn man den Namen einer Datenschicht (Layer) nur rät, statt ihn in der Dokumentation nachzuschlagen.

⚠️ Die folgende Zelle bricht **absichtlich** mit einem Fehler ab.

In [ ]:
url = "https://geoweb1.digistadtdo.de/doris_gdi/geoserver/ALKIS_ADV/ows"
params = {
    "service": "WFS",
    "version": "1.0.0",
    "request": "GetFeature",
    "typeName": "ALKIS_ADV:ALKIS_ADV_FlurstueckMitFehler",
    "outputFormat": "application/json",
}

response = requests.get(url, params=params, timeout=10)
print(response.status_code)

data = response.json()
print(data)

Der Statuscode `200` signalisiert eigentlich Erfolg — trotzdem scheitert `response.json()`. Ein Blick in `response.text` verrät, warum: Bei einem ungültigen Parameter liefert dieser Server keinen Fehlerstatus, sondern eine Fehlermeldung als **XML** statt als JSON zurück:

In [ ]:
print(response.text)

⚠️ Diese API zeigt ein Problem, das dir bei offenen APIs öfter begegnet: Der HTTP-Status allein sagt nicht zuverlässig aus, ob die Antwort brauchbar ist. `response.raise_for_status()` hilft hier nicht — dafür schlägt `response.json()` fehl. Robuster Code prüft deshalb beides:

In [ ]:
params["typeName"] = "ALKIS_ADV:ALKIS_ADV_GebaeudeBauwerk"  # korrekter Name laut GetCapabilities
params["maxFeatures"] = 50  # sonst werden alle Gebäude Dortmunds geladen (>140 MB!)

try:
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.JSONDecodeError:
    print(f"Antwort ist kein gültiges JSON:\n{response.text[:200]}")
except requests.exceptions.RequestException as error:
    print(f"Netzwerkfehler: {error}")
else:
    print(f"{len(data['features'])} Gebäude geladen.")
    print(data["features"][0]["properties"])  # Eigenschaften des ersten Gebäudes

Zwei Lehren aus diesem Beispiel: Erstens lohnt sich bei einer unbekannten API ein Blick in ihre Dokumentation (bzw. hier die `GetCapabilities`-Antwort des WFS-Dienstes) statt Namen zu raten. Zweitens solltest du bei großen offenen Datensätzen die Anfrage serverseitig einschränken (hier über `maxFeatures`), statt riesige Datenmengen ungefiltert zu laden.

<a id="uebungsaufgaben"></a>

## Übungsaufgaben

Die folgenden Aufgaben greifen alle Kapitel des Notebooks noch einmal auf. Unter jeder Aufgabe steht eine Codezelle mit einem Anfang, der vervollständigt werden soll. Die Kommentare `# erwartet: ...` zeigen, welches Ergebnis herauskommen sollte.

Die Aufgaben arbeiten mit der Datei `einkaufsliste.txt` aus diesem Ordner und legen weitere Dateien im selben Ordner an.

### Dateien lesen

#### Aufgabe 1: Einträge zählen

Zähle mit einer `for`-Schleife über das Datei-Objekt, wie viele Einträge `einkaufsliste.txt` enthält. Verwende dabei **nicht** `.readlines()`.

In [ ]:
line_count = 0

with open("einkaufsliste.txt", "r", encoding="utf-8") as shopping_list_file:
    # TODO: über die Zeilen iterieren und line_count erhöhen
    pass

print(f"Die Einkaufsliste hat {line_count} Einträge.")   # erwartet: 5

#### Aufgabe 2: Nummerierte Liste

Gib die Einkaufsliste nummeriert und ohne Leerzeilen aus:

```
1. Äpfel
2. Birnen
3. Brot
...
```

Tipp: `enumerate()` kennst du aus der Session "Kontrollstrukturen". Mit `enumerate(..., start=1)` beginnt die Zählung bei 1.

In [ ]:
with open("einkaufsliste.txt", "r", encoding="utf-8") as shopping_list_file:
    # TODO: Zeilen mit enumerate() durchlaufen und nummeriert ausgeben
    pass

#### Aufgabe 3: Erste Zeile gesondert behandeln

Lies mit `.readline()` nur die erste Zeile und gib sie als `Wichtigster Eintrag: ...` aus. Lies anschließend mit `.readlines()` die **restlichen** Zeilen ein und gib sie als Liste ohne `\n` aus.

Erwartete Ausgabe:
```
Wichtigster Eintrag: Äpfel
Weitere Einträge: ['Birnen', 'Brot', 'Butter', 'Kaffee']
```

Tipp: `.readlines()` liest ab der Stelle weiter, an der `.readline()` aufgehört hat.

In [ ]:
with open("einkaufsliste.txt", "r", encoding="utf-8") as shopping_list_file:
    first_line = ""         # TODO: erste Zeile mit .readline() lesen
    remaining_lines = []    # TODO: restliche Zeilen mit .readlines() lesen

remaining_items = []
# TODO: jede Zeile aus remaining_lines mit .strip() bereinigen und an remaining_items anhängen

print(f"Wichtigster Eintrag: {first_line.strip()}")
print(f"Weitere Einträge: {remaining_items}")

#### Aufgabe 4: Eintrag suchen

Schreibe eine Funktion `contains_item(filename, item)`, die `True` zurückgibt, wenn `item` als eigene Zeile in der Datei steht, sonst `False`.

In [ ]:
def contains_item(filename, item):
    """Return True if item is a line in the file, otherwise False."""
    with open(filename, "r", encoding="utf-8") as file:
        # TODO: Zeilen durchlaufen und bereinigt mit item vergleichen
        pass
    return False


print(contains_item("einkaufsliste.txt", "Brot"))    # erwartet: True
print(contains_item("einkaufsliste.txt", "Milch"))   # erwartet: False

### Dateien schreiben

#### Aufgabe 5: Aufgabenliste speichern

Schreibe die Einträge der Liste `tasks` zeilenweise in eine neue Datei `aufgaben.txt`. Lies die Datei anschließend zur Kontrolle wieder ein und gib den Inhalt aus.

In [ ]:
tasks = ["Wäsche waschen", "Python üben", "Einkaufen"]

# TODO: aufgaben.txt im Modus 'w' öffnen und jede Aufgabe mit "\n" schreiben

# TODO: aufgaben.txt wieder einlesen und ausgeben

#### Aufgabe 6: Aufgaben ergänzen

Schreibe eine Funktion `add_task(filename, task)`, die eine Aufgabe als neue Zeile an das Ende der Datei anhängt. Ergänze damit zwei weitere Aufgaben in `aufgaben.txt` und gib den Inhalt danach aus.

Überlege: Was passiert, wenn du die Zelle mehrmals ausführst?

In [ ]:
def add_task(filename, task):
    """Append task as a new line to the file."""
    # TODO: Datei im passenden Modus öffnen und task schreiben
    pass


add_task("aufgaben.txt", "Steuererklärung")
add_task("aufgaben.txt", "Sport")

# TODO: aufgaben.txt einlesen und ausgeben

#### Aufgabe 7: `'w'` oder `'a'`?

Überlege **vor** dem Ausführen, was am Ende in `modus_test.txt` steht, und notiere deine Vermutung im Kommentar. Führe die Zelle danach aus und vergleiche.

In [ ]:
# Vermutung: TODO

with open("modus_test.txt", "w", encoding="utf-8") as test_file:
    test_file.write("A\n")

with open("modus_test.txt", "w", encoding="utf-8") as test_file:
    test_file.write("B\n")

with open("modus_test.txt", "a", encoding="utf-8") as test_file:
    test_file.write("C\n")

with open("modus_test.txt", "r", encoding="utf-8") as test_file:
    print(test_file.read())

#### Aufgabe 8: Sortierte Einkaufsliste

Lies `einkaufsliste.txt` ein, sortiere die Einträge mit `sorted()` und schreibe sie in eine neue Datei `einkaufsliste_sortiert.txt`. Lies die neue Datei zur Kontrolle wieder ein.

Achte auf die Position von `Äpfel` im Ergebnis. Die Erklärung dazu steht im Notebook "Funktionen" beim Abschnitt zu `sorted()`.

In [ ]:
items = []
# TODO: einkaufsliste.txt einlesen, jede Zeile mit .strip() bereinigen und an items anhängen

sorted_items = items   # TODO: mit sorted() sortieren

# TODO: sorted_items zeilenweise in einkaufsliste_sortiert.txt schreiben

# TODO: einkaufsliste_sortiert.txt einlesen und ausgeben

#### Aufgabe 9: Messwerte auswerten

Die erste Zeile der Codezelle legt eine Datei `messwerte.txt` mit Temperaturen an. Lies die Datei ein und berechne Summe und Durchschnitt der Werte.

Tipp: Aus der Datei kommen Strings. Mit `float()` wird daraus eine Zahl.

In [ ]:
# Testdatei anlegen (nicht verändern)
with open("messwerte.txt", "w", encoding="utf-8") as measurements_file:
    measurements_file.write("21.5\n19.0\n23.25\n20.75\n")

total = 0
count = 0
# TODO: messwerte.txt zeilenweise lesen, jede Zeile in float umwandeln, total und count erhöhen

average = 0   # TODO: Durchschnitt berechnen

print(f"Summe: {total}")          # erwartet: 84.5
print(f"Durchschnitt: {average}")  # erwartet: 21.125

### Fehlerbehandlung

#### Aufgabe 10: Sicheres Einlesen

Schreibe eine Funktion `read_file_safely(filename)`, die den Inhalt einer Datei als String zurückgibt. Existiert die Datei nicht, soll sie die Meldung `Datei <filename> nicht gefunden.` ausgeben und `None` zurückgeben.

In [ ]:
def read_file_safely(filename):
    """Return the file content or None if the file does not exist."""
    # TODO: Datei in try lesen und Inhalt zurückgeben
    # TODO: FileNotFoundError abfangen, Meldung ausgeben und None zurückgeben
    pass


print(read_file_safely("einkaufsliste.txt"))    # erwartet: Inhalt der Einkaufsliste
print(read_file_safely("gibt_es_nicht.txt"))    # erwartet: Meldung und None

#### Aufgabe 11: Ausführungszähler

Baue einen Zähler, der sich merkt, wie oft die Zelle schon ausgeführt wurde — auch über einen Neustart des Notebooks hinweg:

1. Den bisherigen Zählerstand aus `besuche.txt` lesen. Existiert die Datei noch nicht, mit `0` starten.
2. Den Zählerstand um 1 erhöhen.
3. Den neuen Stand in `besuche.txt` schreiben.

Führe die Zelle mehrmals aus.

Tipp: `int()` wandelt den gelesenen String in eine Zahl um, `str()` eine Zahl zurück in einen String.

In [ ]:
counter_filename = "besuche.txt"

visit_count = 0
# TODO: Zählerstand aus counter_filename lesen, bei FileNotFoundError bei 0 bleiben

# TODO: visit_count um 1 erhöhen

# TODO: neuen Zählerstand in counter_filename schreiben

print(f"Diese Zelle wurde {visit_count}-mal ausgeführt.")

### Optional: CSV, JSON und APIs

#### Aufgabe 12: Produkte als CSV

Schreibe die Liste `products` mit `csv.DictWriter` in eine Datei `produkte.csv`. Lies die Datei anschließend mit `csv.DictReader` wieder ein und berechne die Summe aller Preise.

Achtung: Der `DictReader` liefert alle Werte als Strings zurück.

In [ ]:
import csv

products = [
    {"name": "Kaffee", "price": 4.99},
    {"name": "Brot", "price": 2.49},
    {"name": "Käse", "price": 6.79},
]

# TODO: products mit csv.DictWriter in produkte.csv schreiben (newline="" nicht vergessen)

total_price = 0
# TODO: produkte.csv mit csv.DictReader lesen und die Preise aufsummieren

print(f"Gesamtpreis: {round(total_price, 2)}")   # erwartet: 14.27

#### Aufgabe 13: Einstellungen als JSON

Speichere das Dictionary `settings` mit `json.dump()` in einer Datei `einstellungen.json`. Lade die Datei danach mit `json.load()`, ändere `font_size` auf `14`, speichere erneut und gib die neu geladenen Einstellungen aus.

In [ ]:
import json

settings = {"theme": "dark", "font_size": 12, "recent_files": ["bericht.txt", "notizen.txt"]}

# TODO: settings in einstellungen.json speichern

loaded_settings = {}
# TODO: einstellungen.json laden, font_size auf 14 setzen und erneut speichern

# TODO: einstellungen.json noch einmal laden und ausgeben
# erwartet: {'theme': 'dark', 'font_size': 14, 'recent_files': ['bericht.txt', 'notizen.txt']}

#### Aufgabe 14: Wetter für eine andere Stadt

Frage über die Open-Meteo-API die aktuelle Temperatur **und** Windgeschwindigkeit für Berlin ab (Breitengrad `52.52`, Längengrad `13.41`). Fange Netzwerkfehler mit `try`/`except` ab.

Tipp: Mehrere Werte werden beim Parameter `current` durch Komma getrennt angegeben, die Windgeschwindigkeit heißt `wind_speed_10m`.

In [ ]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {}   # TODO: latitude, longitude und current eintragen

# TODO: Anfrage in try senden und mit raise_for_status() prüfen
# TODO: requests.exceptions.RequestException abfangen
# TODO: Temperatur und Windgeschwindigkeit aus data["current"] ausgeben